# Simple RVC - Voice Conversion Demo

This notebook demonstrates how to use [Simple RVC](https://github.com/SawitProject/rvc) for voice conversion in Google Colab.

## What you'll need
- A `.pth` RVC model file (upload to Colab or provide a download URL)
- An input audio file (WAV, MP3, FLAC, etc.)
- Optional: a `.index` file for better voice quality

## Quick Steps
1. **Install** RVC (Step 1)
2. **Upload** your model and audio files (Step 2)
3. **Run** voice conversion (Step 3)
4. **Download** the result (Step 4)

In [ ]:
# @title **1. Install RVC** { display-mode: "form" }
# @markdown This cell installs the RVC package and its dependencies. It takes about 2-3 minutes.
import subprocess

# Install FFmpeg
print("Installing FFmpeg...")
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], capture_output=True)
print("FFmpeg installed.")

# Install RVC from GitHub
print("Installing RVC...")
!pip install git+https://github.com/SawitProject/rvc.git -q

# Verify installation
try:
    from rvc.infer import __version__
    print(f"\nRVC installed successfully! Version: {__version__}")
except ImportError:
    print("\nRVC installed successfully!")

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Using CPU (will be slower).")

In [ ]:
# @title **2. Upload Files** { display-mode: "form" }
# @markdown Upload your model (`.pth`) and input audio file. You can also provide download URLs instead.
import os
from google.colab import files

# Create working directory
os.makedirs("/content/rvc_workspace", exist_ok=True)

upload_method = "upload"  # @param ["upload", "url"]

if upload_method == "upload":
    print("Please upload your model (.pth) and audio files:")
    print("(You can select multiple files at once)\n")
    uploaded = files.upload()
    for filename in uploaded.keys():
        src = filename
        dst = os.path.join("/content/rvc_workspace", filename)
        if src != dst:
            os.rename(src, dst)
        print(f"  Saved: {dst}")
else:
    model_url = ""  # @param {type:"string"}
    audio_url = ""  # @param {type:"string"}
    index_url = ""  # @param {type:"string"}

    if model_url:
        print(f"Downloading model...")
        !wget -q "{model_url}" -O /content/rvc_workspace/model.pth
        print("  Model saved to /content/rvc_workspace/model.pth")
    if audio_url:
        print(f"Downloading audio...")
        !wget -q "{audio_url}" -O /content/rvc_workspace/input.wav
        print("  Audio saved to /content/rvc_workspace/input.wav")
    if index_url:
        print(f"Downloading index...")
        !wget -q "{index_url}" -O /content/rvc_workspace/model.index
        print("  Index saved to /content/rvc_workspace/model.index")

print("\nFiles in workspace:")
!ls -lh /content/rvc_workspace/

In [ ]:
# @title **3. Run Voice Conversion** { display-mode: "form" }
# @markdown Configure the conversion parameters and run the pipeline.
import glob
from rvc.infer.infer import run_inference_script
from rvc.lib.config import Config

# --- File Paths ---
workspace = "/content/rvc_workspace"

# Auto-detect files in workspace
pth_files = glob.glob(os.path.join(workspace, "*.pth"))
audio_files = glob.glob(os.path.join(workspace, "*.wav")) + \
             glob.glob(os.path.join(workspace, "*.mp3")) + \
             glob.glob(os.path.join(workspace, "*.flac"))
index_files = glob.glob(os.path.join(workspace, "*.index"))

# @markdown ### File Paths (auto-detected or manual override)
model_path = pth_files[0] if pth_files else ""  # @param {type:"string"}
input_audio = audio_files[0] if audio_files else ""  # @param {type:"string"}
output_audio = os.path.join(workspace, "output.wav")  # @param {type:"string"}
index_path = index_files[0] if index_files else ""  # @param {type:"string"}

# @markdown ### Voice Conversion Settings
pitch = 0  # @param {type:"slider", min:-24, max:24, step:1}
f0_method = "rmvpe"  # @param ["pm", "dio", "harvest", "yin", "pyin", "swipe", "rmvpe", "rmvpe-legacy", "fcpe", "fcpe-legacy", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "crepe-full", "mangio-crepe-tiny", "mangio-crepe-small", "mangio-crepe-medium", "mangio-crepe-large", "mangio-crepe-full", "djcm"]
index_rate = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
volume_envelope = 1.0  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
protect = 0.5  # @param {type:"slider", min:0.0, max:0.5, step:0.05}
filter_radius = 3  # @param {type:"slider", min:1, max:10, step:1}
hop_length = 64  # @param {type:"slider", min:32, max:256, step:32}

# @markdown ### Advanced Features
enable_autotune = False  # @param {type:"boolean"}
autotune_strength = 1.0  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
enable_clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
enable_formant_shifting = False  # @param {type:"boolean"}
formant_qfrency = 0.8  # @param {type:"slider", min:0.0, max:1.6, step:0.1}
formant_timbre = 0.8  # @param {type:"slider", min:0.0, max:1.6, step:0.1}

# @markdown ### Output Settings
output_format = "wav"  # @param ["wav", "flac", "mp3", "ogg"]
resample_sr = 0  # @param {type:"slider", min:0, max:48000, step:1000}

# Validate inputs
if not model_path or not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")
if not input_audio or not os.path.exists(input_audio):
    raise FileNotFoundError(f"Input audio not found: {input_audio}")

# Update output path with correct extension
if not output_audio.endswith(f".{output_format}"):
    output_audio = os.path.splitext(output_audio)[0] + f".{output_format}"

print("=" * 50)
print("RVC Voice Conversion")
print("=" * 50)
print(f"  Model:      {model_path}")
print(f"  Input:      {input_audio}")
print(f"  Output:     {output_audio}")
print(f"  Pitch:      {pitch:+d} semitones")
print(f"  F0 Method:  {f0_method}")
print(f"  Index:      {index_path or 'None'}")
print(f"  Index Rate: {index_rate}")
print(f"  Autotune:   {'On (strength=' + str(autotune_strength) + ')' if enable_autotune else 'Off'}")
print(f"  Clean:      {'On (strength=' + str(clean_strength) + ')' if enable_clean_audio else 'Off'}")
print("=" * 50)

# Initialize config
config = Config()

# Run inference
print("\nStarting voice conversion...\n")
run_inference_script(
    config=config,
    input_path=input_audio,
    output_path=output_audio,
    pth_path=model_path,
    index_path=index_path if index_path else None,
    pitch=pitch,
    f0_method=f0_method,
    index_rate=index_rate,
    volume_envelope=volume_envelope,
    protect=protect,
    filter_radius=filter_radius,
    hop_length=hop_length,
    export_format=output_format,
    embedder_model="contentvec_base",
    resample_sr=resample_sr,
    f0_autotune=enable_autotune,
    f0_autotune_strength=autotune_strength,
    split_audio=False,
    clean_audio=enable_clean_audio,
    clean_strength=clean_strength,
    formant_shifting=enable_formant_shifting,
    formant_qfrency=formant_qfrency,
    formant_timbre=formant_timbre,
)

if os.path.exists(output_audio):
    file_size = os.path.getsize(output_audio) / 1024 / 1024
    print(f"\nConversion complete! Output: {output_audio} ({file_size:.2f} MB)")
else:
    print("\nConversion may have failed. Check the output above for errors.")

In [ ]:
# @title **4. Play & Download Result** { display-mode: "form" }
# @markdown Listen to the converted audio and download it.
from IPython.display import Audio, display
from google.colab import files

if os.path.exists(output_audio):
    print("Playing converted audio:")
    display(Audio(output_audio))

    print("\nDownloading...")
    files.download(output_audio)
else:
    print("Output file not found. Did the conversion succeed?")

In [ ]:
# @title **5. Batch Conversion** (Optional) { display-mode: "form" }
# @markdown Convert multiple audio files at once by placing them in a directory.
from rvc.infer.cli import VoiceConverter
from rvc.lib.config import Config

batch_input_dir = "/content/rvc_workspace"  # @param {type:"string"}
batch_pitch = 0  # @param {type:"slider", min:-24, max:24, step:1}
batch_f0_method = "rmvpe"  # @param ["pm", "rmvpe", "fcpe", "crepe-large"]

config = Config()
cvt = VoiceConverter(config, model_path, 0)

audio_exts = ("wav", "mp3", "flac", "ogg", "opus", "m4a")
audio_files = [
    f for f in os.listdir(batch_input_dir)
    if f.lower().endswith(audio_exts) and not f.startswith("output")
]

print(f"Found {len(audio_files)} audio file(s) in {batch_input_dir}\n")

for audio_file in audio_files:
    input_path = os.path.join(batch_input_dir, audio_file)
    output_name = os.path.splitext(audio_file)[0] + "_converted.wav"
    output_path = os.path.join(batch_input_dir, output_name)

    print(f"Converting: {audio_file} -> {output_name}")
    cvt.convert_audio(
        audio_input_path=input_path,
        audio_output_path=output_path,
        index_path=index_path if index_path else None,
        embedder_model="contentvec_base",
        pitch=batch_pitch,
        f0_method=batch_f0_method,
        index_rate=0.5,
        volume_envelope=1.0,
        protect=0.5,
        hop_length=64,
        filter_radius=3,
        export_format="wav",
    )
    print(f"  Done!\n")

print("Batch conversion complete!")

---

## Tips & Tricks

- **Pitch**: Use positive values to make the voice higher, negative for lower. Start with small values (+/- 5) and adjust.
- **F0 Method**: `rmvpe` gives the best quality. Use `pm` for fastest results. `crepe-large` for highest accuracy.
- **Index Files**: Using an index file with `index_rate=0.5-0.75` can significantly improve voice similarity.
- **Autotune**: Great for singing, set strength to 0.5-0.8 for natural results.
- **Noise Reduction**: Use if your input audio has background noise. Start with `clean_strength=0.5`.
- **Hybrid F0**: Combine methods like `hybrid[rmvpe+fcpe]` for more robust pitch detection.

For more details, see the [documentation](https://github.com/SawitProject/rvc/blob/main/DOCUMENTATION.md).